# API Basics: Talking to AI Models

This notebook teaches you how to make API calls to AI models. An **API** (Application Programming Interface) lets your code communicate with external services—in this case, OpenAI's GPT models.

## Why Use Code Instead of ChatGPT?

You can already talk to AI through ChatGPT's web interface. So why learn to do it through code? Three reasons:

1. **Scale.** With code, you can loop through hundreds of poems, passages, or documents automatically. Doing that one-by-one in a chat interface would take forever.

2. **Chaining.** You can connect prompts together into workflows: extract characters from a novel, then classify their relationships, then visualize the results—all in one script.

3. **Control.** The API gives you access to settings the web interface doesn't expose, especially **structured outputs**—getting back organized data (like a dictionary) instead of just paragraphs of text.

By the end of this notebook, you will:
- Understand the basic pattern of an API call
- Send questions to an AI model and get responses
- Build reusable functions for text analysis
- See how this scales beyond what's possible in a chat interface

**Prerequisites:** You should have completed `00_python_basics.ipynb` first.

---

## What is an API?

Think of an API as a **messenger**. You write a request ("analyze this poem"), the messenger carries it to the AI service, and brings back the response.

The pattern is always:
1. **Connect** to the service (create a "client")
2. **Send** a request (your question or prompt)
3. **Receive** a response (the AI's answer)

---

## Setup

Your OpenAI API key should be configured in **Codespaces Secrets**. If you haven't done this yet:

1. Go to [github.com/settings/codespaces](https://github.com/settings/codespaces)
2. Scroll to **Secrets**
3. Click **New secret**
4. Name: `OPENAI_API_KEY`, Value: your API key
5. Select this repository (or all repositories)
6. **Rebuild your Codespace** (or restart it) for the secret to take effect

Run this cell to verify everything is set up:

In [1]:
# ============================================================
# SETUP: Run this cell first
# ============================================================

import os

# Check for API key
if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not found!\n\n"
        "Please add it to your Codespaces Secrets:\n"
        "1. Go to: github.com/settings/codespaces\n"
        "2. Add a secret named OPENAI_API_KEY with your key\n"
        "3. Restart your Codespace\n"
    )

print("✓ API key found")

# Import OpenAI
from openai import OpenAI
print("✓ OpenAI library ready")

✓ API key found


✓ OpenAI library ready


---

## Your First API Call

Here's the simplest possible API call. Don't worry about memorizing the syntax—you can always copy and modify this pattern:

In [2]:
from openai import OpenAI

# Create a "client" — this is your connection to the API
client = OpenAI()

# Send a message and get a response
response = client.chat.completions.create(
    model="gpt-4o-mini",  # Which AI model to use
    max_tokens=256,        # Maximum length of response
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is Keats's 'On First Looking into Chapman's Homer' about?"
        }
    ]
)

# The response is a complex object; the actual text is here:
print(response.choices[0].message.content)

Keats's "On First Looking into Chapman's Homer" expresses the profound awe and exhilaration the speaker feels upon discovering the beauty and grandeur of Homer’s epic poetry through Chapman's translation.


### What Just Happened?

1. We **imported** the `OpenAI` class from the `openai` library
2. We created a **client** (the connection to the API)
3. We called `client.chat.completions.create()` with:
   - The **model** we want to use (`gpt-4o-mini` is fast and affordable)
   - A **max_tokens** limit (how long the response can be)
   - A list of **messages** (just one, from the "user")
4. We got back a **response** and printed the text

That's it. Everything else we do with AI in this course is a variation on this pattern.

---

## A Poem to Work With

Let's load a poem so we have something to analyze:

In [3]:
poem = """Much have I travelled in the realms of gold,
And many goodly states and kingdoms seen;
Round many western islands have I been
Which bards in fealty to Apollo hold.
Oft of one wide expanse had I been told
That deep-brow'd Homer ruled as his demesne;
Yet did I never breathe its pure serene
Till I heard Chapman speak out loud and bold:
Then felt I like some watcher of the skies
When a new planet swims into his ken;
Or like stout Cortez when with eagle eyes
He stared at the Pacific—and all his men
Look'd at each other with a wild surmise—
Silent, upon a peak in Darien."""

print(poem)

Much have I travelled in the realms of gold,
And many goodly states and kingdoms seen;
Round many western islands have I been
Which bards in fealty to Apollo hold.
Oft of one wide expanse had I been told
That deep-brow'd Homer ruled as his demesne;
Yet did I never breathe its pure serene
Till I heard Chapman speak out loud and bold:
Then felt I like some watcher of the skies
When a new planet swims into his ken;
Or like stout Cortez when with eagle eyes
He stared at the Pacific—and all his men
Look'd at each other with a wild surmise—
Silent, upon a peak in Darien.


---

## Combining Variables and API Calls

Now let's use an **f-string** to include our poem in the prompt:

In [4]:
# Build a prompt that includes our poem
prompt = f"""Here is a poem:

{poem}

List the 5 most striking images in this poem, one per line."""

# Send it to the AI
response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=256,
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

1. "Much have I travelled in the realms of gold,"  
2. "Like some watcher of the skies / When a new planet swims into his ken;"  
3. "Like stout Cortez when with eagle eyes / He stared at the Pacific"  
4. "Look'd at each other with a wild surmise—"  
5. "Silent, upon a peak in Darien."


---

## Making It Reusable: Functions

Typing out `client.chat.completions.create(...)` every time is tedious. Let's wrap it in a function:

In [5]:
def ask(question, max_tokens=256):
    """Send a question to the AI and return the response text."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

# Now asking questions is simple:
answer = ask("What is the rhyme scheme of a Petrarchan sonnet?")
print(answer)

A Petrarchan sonnet, also known as an Italian sonnet, consists of 14 lines divided into two parts: an octave and a sestet. The rhyme scheme for the octave is typically ABBAABBA, while the sestet can vary, with common patterns being CDCDCD or CDECDE. This structure creates a distinct rhythm and allows for a progression of ideas, often presenting a problem in the octave and a resolution in the sestet.


In [6]:
def analyze_poem(poem_text, question):
    """Ask the AI a question about a specific poem."""
    prompt = f"""Here is a poem:

{poem_text}

{question}"""
    return ask(prompt)

# Try it:
print(analyze_poem(poem, "What is the 'turn' or volta in this sonnet?"))

In this poem, which is John Keats's "On First Looking into Chapman's Homer," the turn, or volta, occurs after the line "Till I heard Chapman speak out loud and bold." This represents a shift in the speaker's experience and perception.

Before this line, the speaker describes his extensive travels and the great literary figures he is familiar with, building a sense of anticipation and longing for a profound experience. The moment he hears Chapman's translation of Homer marks a turning point; it is the moment of realization and awe that transforms his previous experiences. 

Following the volta, the speaker compares his newfound understanding to significant moments of discovery: feeling like a watcher of the skies seeing a new planet or Cortez gazing upon the Pacific. This conveys a powerful sense of revelation and wonder, showcasing the impact of Chapman's work on the speaker's appreciation of literature and art.


In [7]:
# Ask about tone
print(analyze_poem(poem, "Describe the emotional arc of this poem in 2-3 sentences."))

The poem begins with a sense of awe and grandeur, as the speaker reflects on their travels through "realms of gold" and encounters with various cultures and landscapes. This admiration turns into an exhilarating realization when they discover a profound connection to poetry through Chapman's translation of Homer, evoking feelings of wonder and enlightenment. Ultimately, the poem culminates in a moment of shared astonishment, paralleling historical exploration, as the speaker and their companions experience a transcendent awakening to art and beauty.


---

## Understanding the Response Object

The API returns more than just text. Let's look at the full response:

In [8]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=50,
    messages=[{"role": "user", "content": "Say hello in exactly 5 words."}]
)

# The full response object
print("Full response:")
print(response)
print()

# The parts we usually care about:
print(f"Model used: {response.model}")
print(f"Text: {response.choices[0].message.content}")
print(f"Tokens used: {response.usage.total_tokens}")

Full response:
ChatCompletion(id='chatcmpl-D1Ak5Yi6AfSNUYOMoNTteLpT6H9o0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How are you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1769171993, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_c4585b5b9c', usage=CompletionUsage(completion_tokens=7, prompt_tokens=15, total_tokens=22, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

Model used: gpt-4o-mini-2024-07-18
Text: Hello! How are you today?
Tokens used: 22


---

## System Messages: Setting Context

You can include a **system message** to set the AI's role or behavior:

In [9]:
def ask_as_scholar(question, max_tokens=256):
    """Ask a question with a literary scholar persona."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=max_tokens,
        messages=[
            {
                "role": "system",
                "content": "You are a literary scholar specializing in Romantic poetry. Give precise, insightful answers that reference specific textual evidence."
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )
    return response.choices[0].message.content

# Compare the responses:
print("Standard response:")
print(ask("What makes Keats's Chapman's Homer sonnet effective?"))
print()
print("Scholar response:")
print(ask_as_scholar("What makes Keats's Chapman's Homer sonnet effective?"))

Standard response:
John Keats's sonnet "On First Looking into Chapman's Homer" is effective for several reasons, rooted in its emotional depth, vivid imagery, and exploration of literary and personal experience. Here are some key elements that contribute to its effectiveness:

1. **Personal Experience**: The poem captures Keats's exhilaration and profound awe upon discovering George Chapman's translation of Homer's epics. This personal connection to literature resonates with readers, inviting them to reflect on their own transformative encounters with art.

2. **Imagery and Metaphor**: Keats employs rich imagery and metaphors that evoke a deep sense of exploration and discovery. The comparison of reading Chapman’s Homer to a journey of exploration—like that of a traveler discovering new lands—has a universal appeal. Phrases like “the real of realms" and “the shores of the great ocean” illustrate the vastness of the literary world.

3. **Structure and Form**: The sonnet's structure adhe

---

## Exercises

Try these on your own:

In [10]:
# EXERCISE 1: Ask about a different poem
# Replace this with your own poem or passage

my_poem = """
Your text here...
"""

# Ask the AI about it:
print(analyze_poem(my_poem, "What is the central theme of this text?"))

It seems that you might have intended to include a poem for analysis, but the text is missing. Please provide the poem, and I would be happy to help you identify the central theme!


In [11]:
# EXERCISE 2: Create your own specialized function
# Example: a function that identifies literary devices

def find_devices(text):
    """Identify literary devices in a text."""
    prompt = f"""Analyze this text for literary devices (metaphor, simile, alliteration, etc.):

{text}

List each device found with a brief quote as evidence."""
    return ask(prompt, max_tokens=500)

print(find_devices(poem))

This text is an excerpt from John Keats' poem "On First Looking into Chapman's Homer." It employs various literary devices. Here’s an analysis of the devices used, along with corresponding quotes from the text:

1. **Metaphor**: The phrase "realms of gold" serves as a metaphor for the rich and vast experiences gained through literature and exploration. It suggests that the literary world is valuable and expansive.

2. **Alliteration**: The phrase "bards in fealty to Apollo" contains alliteration with the repetition of the initial "b" and "f" sounds, emphasizing the connection between the poets and the patron god of poetry, Apollo.

3. **Imagery**: The image of "one wide expanse" evokes a vast and open space, allowing readers to visualize the enormity of the experiences and knowledge being discussed.

4. **Simile**: "Then felt I like some watcher of the skies / When a new planet swims into his ken" is a simile that compares the speaker's feeling of wonder to that of an astronomer discov

In [ ]:
# EXERCISE 3: Experiment with different questions
# Try asking about: imagery, sound, structure, historical context, etc.

my_question = "Your question here"
print(analyze_poem(poem, my_question))

---

## Summary

| Concept | What It Does | Example |
|---------|--------------|--------|
| Client | Connection to the API | `client = OpenAI()` |
| Messages | The conversation history | `[{"role": "user", "content": "..."}]` |
| System message | Sets the AI's persona | `{"role": "system", "content": "..."}` |
| max_tokens | Limits response length | `max_tokens=256` |
| Response | What comes back | `response.choices[0].message.content` |

The core pattern:
```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": your_prompt}]
)
text = response.choices[0].message.content
```

---

## What's Next?

Now that you can talk to AI models, the next notebooks will show you how to:

1. **Get structured output** — not just paragraphs, but organized data
2. **Define schemas** — tell the AI exactly what format you want
3. **Extract information** — pull specific details from texts systematically

Continue to `01_types_and_containers.ipynb` to learn about data types and structured information.